In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, TensorDataset

# Fix the random seeds so that every run (and every teammate) gets the same results
np.random.seed(0)
torch.manual_seed(0)

# Use the GPU if there is one, otherwise the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# The notebook lives in notebooks/, the data one level up in data/
DATA_DIR = Path("..") / "data"

print("device:", device)

device: cpu


In [3]:
# ImageFolder reads one sub-folder per class and numbers the classes in alphabetical order.
# ToTensor converts each image to a float tensor of shape (3, H, W) with values in [0, 1].
style_set = datasets.ImageFolder(DATA_DIR / "wikiart_style_all", transform=transforms.ToTensor())

print(len(style_set), "images,", len(style_set.classes), "classes")

# Look at the first image to check its shape, type and value range
image, label = style_set[0]
print("image:", tuple(image.shape), image.dtype, "| min", image.min().item(), "max", image.max().item())
print("label:", label, style_set.classes[label])

7448 images, 27 classes
image: (3, 128, 128) torch.float32 | min 0.07450980693101883 max 0.9254902005195618
label: 0 Abstract_Expressionism


In [4]:
# Check that PyTorch numbers the classes like the label_idx column of metadata.csv,
# so that a predicted index always means the same class name
style_meta = pd.read_csv(DATA_DIR / "wikiart_style_all" / "metadata.csv")
meta_classes = (
    style_meta.drop_duplicates("label_idx")  # one row per class
    .sort_values("label_idx")                # in index order: 0, 1, 2, ...
    ["label"]
    .tolist()
)
print("same class order as metadata.csv:", meta_classes == style_set.classes)

same class order as metadata.csv: True


In [5]:
# Same loading and same check as for the styles, on the genre subset
genre_set = datasets.ImageFolder(DATA_DIR / "wikiart_genre_all", transform=transforms.ToTensor())
print(len(genre_set), "images,", len(genre_set.classes), "classes")

genre_meta = pd.read_csv(DATA_DIR / "wikiart_genre_all" / "metadata.csv")
meta_classes = (
    genre_meta.drop_duplicates("label_idx")  # one row per class
    .sort_values("label_idx")                # in index order: 0, 1, 2, ...
    ["label"]
    .tolist()
)
print("same class order as metadata.csv:", meta_classes == genre_set.classes)
print("class 0:", genre_set.classes[0])

3300 images, 11 classes
same class order as metadata.csv: True
class 0: Unknown Genre


In [6]:
from sklearn.model_selection import train_test_split


def split_indices(targets, val_size=0.15, test_size=0.15, seed=0):
    """Split the image indices into train / validation / test sets, keeping the class proportions.

    Args:
        targets: the class index of every image (for example style_set.targets).
        val_size: fraction of all the images used for validation.
        test_size: fraction of all the images used for the final test.
        seed: random seed, so that the split is the same at every run.

    Returns:
        Three arrays of image indices: train, validation and test.
    """
    targets = np.array(targets)
    indices = np.arange(len(targets))

    # First set the test images aside...
    # stratify=...: this is the option that maintains the class proportions
    train_val_idx, test_idx = train_test_split(
        indices, test_size=test_size, stratify=targets, random_state=seed
    )

    # ...then split what is left into train and validation.
    # 15 % of all the images is 15 / 85 = 17.6 % of what is left.
    val_fraction = val_size / (1 - test_size)
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_fraction, stratify=targets[train_val_idx], random_state=seed
    )
    return train_idx, val_idx, test_idx


style_train_idx, style_val_idx, style_test_idx = split_indices(style_set.targets)
genre_train_idx, genre_val_idx, genre_test_idx = split_indices(genre_set.targets)

print("styles: train", len(style_train_idx), "| val", len(style_val_idx), "| test", len(style_test_idx))
print("genres: train", len(genre_train_idx), "| val", len(genre_val_idx), "| test", len(genre_test_idx))

styles: train 5212 | val 1118 | test 1118
genres: train 2309 | val 496 | test 495


In [7]:
def load_as_tensors(dataset, batch_size=256):
    """Read all the images of an ImageFolder dataset into memory, as in TP05.

    Args:
        dataset: an ImageFolder dataset whose transform returns tensors.
        batch_size: number of images read at a time (only affects speed).

    Returns:
        X: float32 tensor of shape (N, 3, H, W), pixel values in [0, 1].
        y: int64 tensor of shape (N,), the class index of each image.
    """
    # Read the images in order (no shuffling), so that X[i] is image number i of the dataset
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    images, labels = [], []
    for xb, yb in loader:
        images.append(xb)
        labels.append(yb)
    # Glue all the batches together into one tensor
    return torch.cat(images), torch.cat(labels)


X_style, y_style = load_as_tensors(style_set)
X_genre, y_genre = load_as_tensors(genre_set)
print("styles:", tuple(X_style.shape), "| genres:", tuple(X_genre.shape))

styles: (7448, 3, 128, 128) | genres: (3300, 3, 128, 128)


In [8]:
def standardise(X, train_idx):
    """Standardise the images in place, with the statistics of the training images only.

    Each colour channel (red, green, blue) gets its own mean and standard deviation,
    computed on the training images only. They are then applied to all the images,
    so that the validation and test images stay "unseen".

    Args:
        X: float tensor of shape (N, 3, H, W), modified in place.
        train_idx: indices of the training images.

    Returns:
        mean, std: tensors of shape (1, 3, 1, 1), to keep for standardising new images later.
    """
    X_train = X[torch.as_tensor(train_idx)]  # as_tensor: turn the NumPy indices into a tensor
    # Average over the images (dim 0), the rows (dim 2) and the columns (dim 3): one value per channel
    mean = X_train.mean(dim=(0, 2, 3), keepdim=True)
    std = X_train.std(dim=(0, 2, 3), keepdim=True)
    # In place (trailing underscore): no second copy of the whole dataset in memory
    X.sub_(mean).div_(std)
    return mean, std


style_mean, style_std = standardise(X_style, style_train_idx)
genre_mean, genre_std = standardise(X_genre, genre_train_idx)

# Check: on the training images, each channel should now have mean 0 and std 1
train_check = X_style[torch.as_tensor(style_train_idx)]
print("style train mean per channel:", train_check.mean(dim=(0, 2, 3)).round(decimals=3))
print("style train std per channel: ", train_check.std(dim=(0, 2, 3)).round(decimals=3))

style train mean per channel: tensor([0., 0., -0.])
style train std per channel:  tensor([1., 1., 1.])


In [9]:
BATCH_SIZE = 64  # images per mini-batch, as in TP06


def make_loaders(X, y, train_idx, val_idx, test_idx, batch_size=BATCH_SIZE):
    """Build the train, validation and test DataLoaders of one subset.

    Subset only stores the list of indices, so no image is copied in memory.

    Args:
        X: standardised images, tensor of shape (N, 3, H, W).
        y: class indices, tensor of shape (N,).
        train_idx, val_idx, test_idx: indices of the images of each set.
        batch_size: number of training images per mini-batch.

    Returns:
        train_loader, val_loader, test_loader.
    """
    full_set = TensorDataset(X, y)
    # Shuffle the training images at every epoch, so that the mini-batches always differ
    train_loader = DataLoader(Subset(full_set, train_idx), batch_size=batch_size, shuffle=True)
    # No shuffling for evaluation; bigger batches are faster since no gradient is computed
    val_loader = DataLoader(Subset(full_set, val_idx), batch_size=1000)
    test_loader = DataLoader(Subset(full_set, test_idx), batch_size=1000)
    return train_loader, val_loader, test_loader


style_train_loader, style_val_loader, style_test_loader = make_loaders(
    X_style, y_style, style_train_idx, style_val_idx, style_test_idx
)
genre_train_loader, genre_val_loader, genre_test_loader = make_loaders(
    X_genre, y_genre, genre_train_idx, genre_val_idx, genre_test_idx
)

# Take one training mini-batch to check the shapes
xb, yb = next(iter(style_train_loader))
print("one batch:", tuple(xb.shape), tuple(yb.shape), "| batches per epoch:", len(style_train_loader))

one batch: (64, 3, 128, 128) (64,) | batches per epoch: 82
